In [3]:
from google.colab import files
uploaded = files.upload()

Saving ev_charging_dataset.xlsx to ev_charging_dataset.xlsx


In [6]:
import pandas as pd

all_sheets = pd.read_excel('ev_charging_dataset.xlsx', sheet_name=None)
sessions_df = all_sheets['sessions']
stations_df = all_sheets['stations']

print(sessions_df.shape)
print(stations_df.shape)

(294024, 15)
(33, 5)


In [7]:
regional_demand = sessions_df.groupby('station_region').agg(
    total_sessions=('session_id', 'count'),
    unique_customers=('customer_id', 'nunique')
).reset_index()

regional_supply = stations_df.groupby('region')['station_id'].count().reset_index()
regional_supply.columns = ['region', 'station_count']

print(regional_demand)
print(regional_supply)

       station_region  total_sessions  unique_customers
0              London           40760              5939
1            Midlands           50598              6369
2          North East           30439              5646
3          North West           65789              6667
4            Scotland           20242              4891
5          South East           25168              5161
6          South West           30401              5675
7               Wales           10159              3721
8  Yorkshire & Humber           20468              4875
               region  station_count
0              London              5
1            Midlands              5
2          North East              3
3          North West              7
4            Scotland              3
5          South East              4
6          South West              3
7               Wales              1
8  Yorkshire & Humber              2


In [8]:
gap_analysis = regional_demand.merge(regional_supply, left_on='station_region', right_on='region')

gap_analysis['sessions_per_station'] = gap_analysis['total_sessions'] / gap_analysis['station_count']
gap_analysis['customers_per_station'] = gap_analysis['unique_customers'] / gap_analysis['station_count']

print(gap_analysis[['station_region', 'total_sessions', 'station_count', 'sessions_per_station', 'customers_per_station']].sort_values('sessions_per_station', ascending=False))

       station_region  total_sessions  station_count  sessions_per_station  \
8  Yorkshire & Humber           20468              2          10234.000000   
7               Wales           10159              1          10159.000000   
2          North East           30439              3          10146.333333   
6          South West           30401              3          10133.666667   
1            Midlands           50598              5          10119.600000   
3          North West           65789              7           9398.428571   
0              London           40760              5           8152.000000   
4            Scotland           20242              3           6747.333333   
5          South East           25168              4           6292.000000   

   customers_per_station  
8            2437.500000  
7            3721.000000  
2            1882.000000  
6            1891.666667  
1            1273.800000  
3             952.428571  
0            1187.800000  
4  

In [9]:
for col in ['sessions_per_station', 'customers_per_station']:
    min_val = gap_analysis[col].min()
    max_val = gap_analysis[col].max()
    gap_analysis[col + '_score'] = ((gap_analysis[col] - min_val) / (max_val - min_val)) * 100

gap_analysis['expansion_priority_score'] = gap_analysis[
    ['sessions_per_station_score', 'customers_per_station_score']
].mean(axis=1)

print(gap_analysis[['station_region', 'expansion_priority_score']].sort_values('expansion_priority_score', ascending=False))

       station_region  expansion_priority_score
7               Wales                 99.048706
8  Yorkshire & Humber                 76.820175
6          South West                 65.689885
2          North East                 65.675969
1            Midlands                 54.352881
3          North West                 39.401682
0              London                 27.842859
4            Scotland                 18.018272
5          South East                  6.101006


In [10]:
gap_analysis.to_csv('gap_analysis.csv', index=False)

from google.colab import files
files.download('gap_analysis.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>